# GlaucomAI
### A clearer view. An earlier signal.
**Train on Kaggle data → evaluate on held-out images → try the model here.**

[GitHub source](https://github.com/Cxsmxnaut/GlaucomAI) ·
[Kaggle dataset](https://www.kaggle.com/datasets/deathtrooper/glaucoma-dataset-eyepacs-airogs-light-v2/versions/4)

This notebook includes all project source code. You do not need to clone a repository or run a web server.
In Colab, choose **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.
GPU availability depends on your Colab account. Download and training time depend on the assigned runtime.

**Research prototype, not a diagnosis.** The classes are **non-referable** and **referable glaucoma**.
A non-referable result does not mean the eye is healthy. No clinical performance is promised.

## 1. Install the notebook tools
Keep Colab's preinstalled GPU-enabled PyTorch and Torchvision. This installs dataset access,
the upload controls, and plotting tools. No API keys belong in notebook cells or GitHub.

In [ ]:
import subprocess
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "kagglehub>=0.3.13,<1", "ipywidgets>=7.7,<9", "matplotlib>=3.9,<4", "Pillow>=11,<13"], check=True)
import torch
import torchvision
print("PyTorch:", torch.__version__, "| Torchvision:", torchvision.__version__)
print("Training device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (slower)")

## 2. Load the project
The next cell writes the same Python files tested in GitHub into this runtime's working directory.
The notebook is generated from those files, so the notebook and repository stay in sync.

In [ ]:
# @title Initialize the GlaucomAI project
import json
import os
from pathlib import Path
PROJECT = Path('/content/GlaucomAI') if Path('/content').exists() else Path.cwd() / '.glaucomai-notebook'
PROJECT.mkdir(parents=True, exist_ok=True)
SOURCES = json.loads('{"clinical_audit.py": "\\"\\"\\"Audit locked research checkpoints in Colab without selecting or tuning them.\\"\\"\\"\\n\\nimport argparse\\nfrom hashlib import sha256\\nfrom io import BytesIO\\nimport json\\nfrom pathlib import Path\\n\\nfrom PIL import Image\\nimport torch\\n\\nfrom glaucoma.multidata import HANDLES, RecordDataset\\nfrom glaucoma.pipeline import decode_image, load_model, preprocessing\\nfrom glaucoma.statistics import cluster_bootstrap, score_statistics\\nfrom train import extract_features\\n\\n\\ndef main():\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument(\\"--run\\", type=Path, default=Path(\\"sequential_run\\"))\\n    args = parser.parse_args()\\n    if not torch.cuda.is_available():\\n        raise RuntimeError(\\"Run this audit in the connected Colab GPU runtime.\\")\\n    torch.set_num_threads(2)\\n    device = torch.device(\\"cuda\\")\\n    source = json.loads((args.run / \\"results.json\\").read_text())\\n    rows = json.loads((args.run / \\"manifest.json\\").read_text())\\n    rows = [row for row in rows if row[\\"split\\"] == \\"test\\"]\\n    base = load_model(args.run / source[\\"stages\\"][0][\\"checkpoint\\"]).model.to(device)\\n    features = extract_features(base, RecordDataset(rows), device, 32)\\n    backbone = {key: value.cpu().clone() for key, value in base.features.state_dict().items()}\\n    report = {\\"training_run_sha256\\": sha256((args.run / \\"results.json\\").read_bytes()).hexdigest(),\\n              \\"audit_device\\": torch.cuda.get_device_name(0), \\"stages\\": [],\\n              \\"clinical_release_allowed\\": False,\\n              \\"clinical_release_reason\\": \\"Research data and engineering tests cannot establish clinical safety. Intended-use approval, independent multisite clinical evidence, image-quality validation, workflow validation, and jurisdiction review remain absent.\\",\\n              \\"test_reuse_note\\": \\"These test results were already inspected after the locked sequential run. They are development benchmarks for future experiments; they cannot remain an unseen confirmatory test.\\",\\n              \\"negative_control_note\\": \\"Solid-color images are raw-model negative controls, bypassing the application\'s input guard; they are not a validated retinal image-quality test. See the separate input guard audit for application abstention.\\"}\\n    for stage in source[\\"stages\\"]:\\n        path = args.run / stage[\\"checkpoint\\"]\\n        if sha256(path.read_bytes()).hexdigest() != stage[\\"checkpoint_sha256\\"]:\\n            raise ValueError(\\"Checkpoint changed after training results were locked.\\")\\n        loaded = load_model(path)\\n        model = loaded.model.to(device).eval()\\n        if any(not torch.equal(value.cpu(), backbone[key]) for key, value in model.features.state_dict().items()):\\n            raise ValueError(\\"Shared feature audit requires identical frozen backbones.\\")\\n        with torch.inference_mode():\\n            scores = model.classifier(features.tensors[0].to(device)).softmax(1)[:, 1].cpu().tolist()\\n        entry = {\\"stage\\": stage[\\"stage\\"], \\"checkpoint_sha256\\": stage[\\"checkpoint_sha256\\"], \\"datasets\\": {}}\\n        for domain in HANDLES:\\n            indices = [i for i, row in enumerate(rows) if row[\\"dataset\\"] == domain]\\n            labels = [rows[i][\\"label\\"] for i in indices]\\n            probabilities = [scores[i] for i in indices]\\n            metrics = score_statistics(labels, probabilities, stage[\\"threshold\\"])\\n            # Recomputed outcomes must agree exactly with the locked confusion counts.\\n            for key in (\\"samples\\", \\"true_positive\\", \\"true_negative\\", \\"false_positive\\", \\"false_negative\\"):\\n                if metrics[key] != stage[\\"test_metrics\\"][domain][key]:\\n                    raise ValueError(\\"Audit disagrees with a locked test outcome.\\")\\n            if domain == \\"papila\\":\\n                metrics[\\"patient_bootstrap_95\\"] = cluster_bootstrap(labels, probabilities,\\n                    [rows[i][\\"group\\"] for i in indices], stage[\\"threshold\\"])\\n            else:\\n                metrics[\\"patient_bootstrap_95\\"] = None\\n                metrics[\\"patient_interval_status\\"] = \\"Unavailable: no independently verified patient identifiers.\\"\\n            entry[\\"datasets\\"][domain] = metrics\\n        controls = []\\n        for color in ((0, 0, 0), (255, 255, 255), (128, 128, 128), (220, 40, 20), (0, 180, 70)):\\n            image = Image.new(\\"RGB\\", (512, 512), color)\\n            buffer = BytesIO()\\n            image.save(buffer, format=\\"PNG\\")\\n            with decode_image(buffer.getvalue(), \\"negative_control.png\\") as decoded:\\n                tensor = preprocessing()(decoded).unsqueeze(0).to(device)\\n            with torch.inference_mode():\\n                score = model(tensor).softmax(1)[0, 1].item()\\n            controls.append({\\"control\\": f\\"solid_rgb{color}\\", \\"positive_score\\": score,\\n                             \\"format_validation_passed\\": True,\\n                             \\"above_threshold\\": score >= stage[\\"threshold\\"], \\"input_abstention\\": False})\\n        entry[\\"non_retinal_negative_controls\\"] = controls\\n        report[\\"stages\\"].append(entry)\\n        print(\\"Audited\\", stage[\\"stage\\"], flush=True)\\n    (args.run / \\"clinical_audit.json\\").write_text(json.dumps(report, indent=2))\\n    print(\\"CLINICAL AUDIT COMPLETE\\", flush=True)\\n    print(json.dumps(report, indent=2), flush=True)\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "evaluate.py": "\\"\\"\\"Evaluate a locked model on held-out images; report real CPU latency.\\"\\"\\"\\n\\nimport argparse\\nimport hashlib\\nimport json\\nfrom pathlib import Path\\nimport statistics\\nimport time\\n\\nimport torch\\n\\nfrom glaucoma.data import FundusDataset, check_split_overlap\\nfrom glaucoma.metrics import binary_metrics\\nfrom glaucoma.pipeline import load_model\\n\\n\\ndef main():\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument(\\"--data\\", type=Path, default=Path(\\"data/prepared\\"))\\n    parser.add_argument(\\"--model\\", type=Path, default=Path(\\"models/best_model.pth\\"))\\n    parser.add_argument(\\"--output\\", type=Path, default=Path(\\"reports/evaluation.json\\"))\\n    args = parser.parse_args()\\n    torch.set_num_threads(2)\\n    train, val, test = [FundusDataset(args.data / split) for split in (\\"train\\", \\"val\\", \\"test\\")]\\n    check_split_overlap(train, val, test)\\n    loaded = load_model(args.model)\\n    labels, scores, latencies = [], [], []\\n    with torch.inference_mode():\\n        # Report cold model execution separately from warmed per-image pipeline.\\n        warmup_start = time.perf_counter()\\n        loaded.model(test[0][0].unsqueeze(0))\\n        warmup_ms = (time.perf_counter() - warmup_start) * 1000\\n        for index in range(len(test)):\\n            start = time.perf_counter()\\n            image, label = test[index]\\n            score = loaded.model(image.unsqueeze(0)).softmax(dim=1)[0, 1].item()\\n            latencies.append((time.perf_counter() - start) * 1000)\\n            labels.append(label)\\n            scores.append(score)\\n    metrics = binary_metrics(labels, scores, loaded.threshold)\\n    sorted_times = sorted(latencies)\\n    p95 = sorted_times[min(len(sorted_times) - 1, int(0.95 * len(sorted_times)))]\\n    report = {\\n        \\"split\\": \\"test\\", \\"threshold\\": loaded.threshold, \\"metrics\\": metrics,\\n        \\"checkpoint_sha256\\": hashlib.sha256(args.model.read_bytes()).hexdigest(),\\n        \\"warmup_ms\\": warmup_ms, \\"median_pipeline_ms\\": statistics.median(latencies),\\n        \\"p95_pipeline_ms\\": p95,\\n        \\"sensitivity_target_met\\": metrics[\\"sensitivity\\"] >= 0.88,\\n        \\"latency_target_met\\": p95 <= 2000,\\n        \\"note\\": \\"Latency includes file decoding, preprocessing and CPU forward pass, not browser transport or UI rendering. Patient separation must be verified by the dataset owner. Model scores are not calibrated clinical probabilities.\\",\\n    }\\n    args.output.parent.mkdir(parents=True, exist_ok=True)\\n    args.output.write_text(json.dumps(report, indent=2))\\n    print(json.dumps(report, indent=2))\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "glaucoma/__init__.py": "\\"\\"\\"GlaucomAI research prototype. No clinical validation is implied.\\"\\"\\"\\n", "glaucoma/data.py": "\\"\\"\\"Explicit class order; identical files cannot cross dataset splits.\\"\\"\\"\\n\\nfrom hashlib import sha256\\nfrom pathlib import Path\\n\\nfrom torch.utils.data import Dataset\\n\\nfrom glaucoma.pipeline import CLASSES, decode_image, preprocessing\\n\\n\\nclass FundusDataset(Dataset):\\n    def __init__(self, root):\\n        self.samples = []\\n        self.transform = preprocessing()\\n        for label, name in enumerate(CLASSES):\\n            folder = Path(root) / name\\n            paths = sorted(p for p in folder.rglob(\\"*\\")\\n                           if p.is_file() and p.suffix.lower() in {\\".png\\", \\".jpg\\", \\".jpeg\\"})\\n            if not paths:\\n                raise ValueError(f\\"No PNG/JPEG images found in {folder}.\\")\\n            self.samples.extend((p, label) for p in paths)\\n\\n    def __len__(self):\\n        return len(self.samples)\\n\\n    def __getitem__(self, index):\\n        path, label = self.samples[index]\\n        return self.transform(decode_image(path.read_bytes(), path.name)), label\\n\\n\\ndef check_split_overlap(*datasets):\\n    seen = {}\\n    for split, dataset in enumerate(datasets):\\n        for path, _ in dataset.samples:\\n            digest = sha256(path.read_bytes()).hexdigest()\\n            if digest in seen and seen[digest][0] != split:\\n                raise ValueError(f\\"Duplicate image crosses dataset splits: {path} and {seen[digest][1]}\\")\\n            seen[digest] = (split, path)\\n", "glaucoma/download.py": "\\"\\"\\"Fetch a published checkpoint and verify its exact SHA-256 before loading.\\"\\"\\"\\n\\nfrom hashlib import sha256\\nfrom pathlib import Path\\nimport urllib.request\\n\\n\\ndef download_checkpoint(url: str, expected_sha256: str, destination=\\"models/best_model.pth\\"):\\n    if not url.startswith(\\"https://github.com/Cxsmxnaut/GlaucomAI/releases/download/\\"):\\n        raise ValueError(\\"Only this project\'s HTTPS release URLs are supported.\\")\\n    if len(expected_sha256) != 64 or any(c not in \\"0123456789abcdef\\" for c in expected_sha256):\\n        raise ValueError(\\"Expected a lowercase SHA-256 checksum.\\")\\n    destination = Path(destination)\\n    if destination.exists():\\n        if sha256(destination.read_bytes()).hexdigest() == expected_sha256:\\n            return destination\\n        raise ValueError(\\"A different checkpoint already exists. Choose a new path or remove it explicitly.\\")\\n    destination.parent.mkdir(parents=True, exist_ok=True)\\n    temporary = destination.with_suffix(destination.suffix + \\".part\\")\\n    digest = sha256()\\n    try:\\n        request = urllib.request.Request(url, headers={\\"User-Agent\\": \\"GlaucomAI-research-demo\\"})\\n        total = 0\\n        with urllib.request.urlopen(request, timeout=60) as response, temporary.open(\\"wb\\") as target:\\n            while chunk := response.read(1024 * 1024):\\n                total += len(chunk)\\n                if total > 100 * 1024 * 1024:\\n                    raise ValueError(\\"Unexpected checkpoint size. Download stopped.\\")\\n                digest.update(chunk)\\n                target.write(chunk)\\n        if digest.hexdigest() != expected_sha256:\\n            raise ValueError(\\"Checkpoint checksum mismatch. The downloaded file was discarded.\\")\\n        temporary.replace(destination)\\n    finally:\\n        temporary.unlink(missing_ok=True)\\n    return destination\\n", "glaucoma/metrics.py": "\\"\\"\\"Small, auditable metrics without an additional ML dependency.\\"\\"\\"\\n\\nimport math\\n\\n\\ndef binary_metrics(labels, probabilities, threshold=0.5):\\n    if len(labels) != len(probabilities) or not labels:\\n        raise ValueError(\\"Provide equally sized, non-empty labels and scores.\\")\\n    if not 0 < threshold < 1:\\n        raise ValueError(\\"Threshold must be between zero and one.\\")\\n    if any(y not in (0, 1) for y in labels):\\n        raise ValueError(\\"Labels must be binary.\\")\\n    if any(not math.isfinite(p) or not 0 <= p <= 1 for p in probabilities):\\n        raise ValueError(\\"Scores must be finite probabilities.\\")\\n    tp = sum(y == 1 and p >= threshold for y, p in zip(labels, probabilities))\\n    tn = sum(y == 0 and p < threshold for y, p in zip(labels, probabilities))\\n    fp = sum(y == 0 and p >= threshold for y, p in zip(labels, probabilities))\\n    fn = sum(y == 1 and p < threshold for y, p in zip(labels, probabilities))\\n    return {\\n        \\"samples\\": len(labels), \\"true_positive\\": tp, \\"true_negative\\": tn,\\n        \\"false_positive\\": fp, \\"false_negative\\": fn,\\n        \\"sensitivity\\": tp / (tp + fn) if tp + fn else None,\\n        \\"specificity\\": tn / (tn + fp) if tn + fp else None,\\n        \\"accuracy\\": (tp + tn) / len(labels),\\n    }\\n\\n\\ndef select_threshold(labels, probabilities, target=0.88):\\n    \\"\\"\\"Choose on validation only; maximize specificity subject to target recall.\\"\\"\\"\\n    if set(labels) != {0, 1} or not 0 < target <= 1:\\n        raise ValueError(\\"Validation needs both classes and a target in (0, 1].\\")\\n    binary_metrics(labels, probabilities)\\n    candidates = sorted({0.001, *[p for p in probabilities if 0 < p < 1]})\\n    feasible = [t for t in candidates\\n                if binary_metrics(labels, probabilities, t)[\\"sensitivity\\"] >= target]\\n    if not feasible:\\n        raise ValueError(\\"No usable threshold meets the validation recall target.\\")\\n    return max(feasible, key=lambda t: (binary_metrics(labels, probabilities, t)[\\"specificity\\"], t))\\n", "glaucoma/multidata.py": "\\"\\"\\"Explicit adapters and locked splits for the sequential research experiment.\\"\\"\\"\\n\\nfrom collections import Counter, defaultdict\\nfrom hashlib import sha256\\nfrom pathlib import Path\\nimport random\\nimport re\\n\\nfrom torch.utils.data import Dataset\\n\\nfrom glaucoma.pipeline import CLASSES, decode_image, preprocessing\\n\\nSEED = 20261107\\nHANDLES = {\\n    \\"eyepacs\\": \\"deathtrooper/glaucoma-dataset-eyepacs-airogs-light-v2/versions/4\\",\\n    \\"acrima\\": \\"orvile/acrima-glaucoma-assessment-using-fundus-images/versions/1\\",\\n    \\"papila\\": \\"orvile/papila-retinal-fundus-images/versions/1\\",\\n}\\n\\n\\ndef locked_group_split(rows, seed=SEED):\\n    \\"\\"\\"Stratify groups by maximum label; no patient can cross splits.\\"\\"\\"\\n    groups = defaultdict(list)\\n    for row in rows:\\n        groups[row[\\"group\\"]].append(row)\\n    strata = defaultdict(list)\\n    for group, members in groups.items():\\n        strata[max(row[\\"label\\"] for row in members)].append(group)\\n    rng = random.Random(seed)\\n    for label in sorted(strata):\\n        keys = sorted(strata[label])\\n        rng.shuffle(keys)\\n        n_test = max(1, round(len(keys) * .15))\\n        n_val = max(1, round(len(keys) * .15))\\n        if len(keys) - n_test - n_val < 1:\\n            raise ValueError(\\"Too few groups for train/validation/test.\\")\\n        for i, group in enumerate(keys):\\n            split = \\"test\\" if i < n_test else \\"val\\" if i < n_test + n_val else \\"train\\"\\n            for row in groups[group]:\\n                row[\\"split\\"] = split\\n    return rows\\n\\n\\ndef acrima_rows(root):\\n    rows = []\\n    for path in sorted(Path(root).rglob(\\"*\\")):\\n        if path.suffix.lower() not in {\\".jpg\\", \\".jpeg\\", \\".png\\"}:\\n            continue\\n        match = re.fullmatch(r\\"Im(\\\\d{3})(_g)?_ACRIMA\\", path.stem)\\n        if not match:\\n            raise ValueError(f\\"Unrecognized ACRIMA image: {path.name}\\")\\n        rows.append({\\"path\\": str(path), \\"label\\": int(bool(match[2])),\\n                     \\"group\\": match[1], \\"dataset\\": \\"acrima\\"})\\n    if Counter(r[\\"label\\"] for r in rows) != {0: 309, 1: 396}:\\n        raise ValueError(\\"ACRIMA must contain 309 normal and 396 glaucoma images.\\")\\n    return locked_group_split(rows)\\n\\n\\ndef papila_rows(root):\\n    # Original paper, Data Records: 0 healthy, 1 glaucoma, 2 suspicious.\\n    # Suspects are excluded, never relabeled as healthy.\\n    import openpyxl\\n\\n    root = Path(root)\\n    images = {p.name: p for p in root.rglob(\\"*.jpg\\") if p.parent.name == \\"FundusImages\\"}\\n    rows, counts = [], Counter()\\n    for eye in (\\"od\\", \\"os\\"):\\n        files = list(root.rglob(f\\"patient_data_{eye}.xlsx\\"))\\n        if len(files) != 1:\\n            raise ValueError(f\\"Expected exactly one PAPILA {eye} clinical table.\\")\\n        workbook = openpyxl.load_workbook(files[0], read_only=True, data_only=True)\\n        try:\\n            table = list(workbook.active.values)\\n        finally:\\n            workbook.close()\\n        # Header names and diagnosis column are checked, not inferred from counts.\\n        header_index = next((i for i, row in enumerate(table)\\n                             if any(str(v).strip().lower() == \\"diagnosis\\" for v in row)), None)\\n        if header_index is None:\\n            raise ValueError(\\"PAPILA diagnosis header not found.\\")\\n        header = [str(v).strip().lower() for v in table[header_index]]\\n        diagnosis = header.index(\\"diagnosis\\")\\n        for row in table[header_index + 1:]:\\n            identifier = row[0]\\n            if identifier is None:\\n                continue\\n            if str(identifier).strip() == \\"ID\\" and row[diagnosis] is None:\\n                continue\\n            match = re.fullmatch(r\\"(?:#|RET)?0*(\\\\d+)\\", str(identifier).strip())\\n            if not match or row[diagnosis] not in (0, 1, 2):\\n                raise ValueError(\\"Unexpected PAPILA patient identifier or diagnosis.\\")\\n            number = int(match[1])\\n            name = f\\"RET{number:03d}{eye.upper()}.jpg\\"\\n            if name not in images:\\n                raise ValueError(f\\"Missing PAPILA image {name}.\\")\\n            label = int(row[diagnosis])\\n            counts[label] += 1\\n            if label == 2:\\n                continue\\n            rows.append({\\"path\\": str(images[name]), \\"label\\": label,\\n                         \\"group\\": f\\"RET{number:03d}\\", \\"dataset\\": \\"papila\\"})\\n    if counts != {0: 333, 1: 87, 2: 68}:\\n        raise ValueError(f\\"Unexpected PAPILA label counts: {counts}\\")\\n    return locked_group_split(rows)\\n\\n\\ndef eyepacs_rows(prepared):\\n    rows = []\\n    for split in (\\"train\\", \\"val\\", \\"test\\"):\\n        for label, name in enumerate(CLASSES):\\n            for path in sorted((Path(prepared) / split / name).glob(\\"*\\")):\\n                if path.is_file():\\n                    rows.append({\\"path\\": str(path), \\"label\\": label, \\"split\\": split,\\n                                 \\"group\\": path.stem, \\"dataset\\": \\"eyepacs\\"})\\n    if Counter(r[\\"split\\"] for r in rows) != {\\"train\\": 8000, \\"val\\": 770, \\"test\\": 770}:\\n        raise ValueError(\\"Unexpected EyePACS publisher split counts.\\")\\n    return rows\\n\\n\\ndef audit_rows(rows):\\n    \\"\\"\\"Reject byte or decoded-pixel duplicates across splits or source datasets.\\"\\"\\"\\n    seen, groups = {}, {}\\n    for i, row in enumerate(rows):\\n        path = Path(row[\\"path\\"])\\n        raw = path.read_bytes()\\n        with decode_image(raw, path.name) as image:\\n            pixel_hash = sha256(str(image.size).encode() + image.tobytes()).hexdigest()\\n        for kind, digest in ((\\"file\\", sha256(raw).hexdigest()), (\\"pixels\\", pixel_hash)):\\n            key = (kind, digest)\\n            if key in seen and seen[key][\\"path\\"] != row[\\"path\\"]:\\n                raise ValueError(f\\"Duplicate image in experiment: {path.name} / {Path(seen[key][\'path\']).name}\\")\\n            seen[key] = row\\n        row[\\"sha256\\"] = sha256(raw).hexdigest()\\n        key = (row[\\"dataset\\"], row[\\"group\\"])\\n        if key in groups and groups[key] != row[\\"split\\"]:\\n            raise ValueError(\\"Patient/group crosses data splits.\\")\\n        groups[key] = row[\\"split\\"]\\n        if (i + 1) % 1000 == 0:\\n            print(f\\"Audited {i + 1}/{len(rows)} images\\", flush=True)\\n    return dict(Counter(f\\"{r[\'dataset\']}/{r[\'split\']}/{r[\'label\']}\\" for r in rows))\\n\\n\\nclass RecordDataset(Dataset):\\n    def __init__(self, rows):\\n        self.rows = rows\\n        self.transform = preprocessing()\\n\\n    def __len__(self):\\n        return len(self.rows)\\n\\n    def __getitem__(self, index):\\n        row = self.rows[index]\\n        path = Path(row[\\"path\\"])\\n        with decode_image(path.read_bytes(), path.name) as image:\\n            return self.transform(image), row[\\"label\\"]\\n\\n\\ndef replay_indices(rows, current, previous, seed=SEED):\\n    \\"\\"\\"All new training data plus at most its size in earlier training examples.\\"\\"\\"\\n    new = [i for i, row in enumerate(rows) if row[\\"dataset\\"] == current and row[\\"split\\"] == \\"train\\"]\\n    rng = random.Random(seed)\\n    replay = []\\n    if previous:\\n        quota = max(1, len(new) // (len(previous) * 2))\\n        for dataset in previous:\\n            for label in (0, 1):\\n                pool = [i for i, row in enumerate(rows) if row[\\"dataset\\"] == dataset\\n                        and row[\\"split\\"] == \\"train\\" and row[\\"label\\"] == label]\\n                rng.shuffle(pool)\\n                replay.extend(pool[:quota])\\n    return new + replay\\n", "glaucoma/multitask.py": "\\"\\"\\"Research-only task-specific heads; never interchangeable clinical outputs.\\"\\"\\"\\n\\nfrom hashlib import sha256\\n\\nimport torch\\nfrom torch import nn\\n\\nfrom glaucoma.pipeline import PREPROCESSING, build_model\\n\\nTASKS = (\\"eyepacs\\", \\"acrima\\", \\"papila\\")\\nARCHITECTURE = \\"efficientnet_b0_separate_tasks_v1\\"\\nTASK_CLASSES = {\\"eyepacs\\": [\\"non_referable\\", \\"referable\\"],\\n                \\"acrima\\": [\\"normal\\", \\"glaucoma\\"], \\"papila\\": [\\"healthy\\", \\"glaucoma\\"]}\\n\\n\\nclass SeparateTaskModel(nn.Module):\\n    def __init__(self, baseline=None):\\n        super().__init__()\\n        baseline = build_model() if baseline is None else baseline\\n        self.features = baseline.features\\n        self.avgpool = baseline.avgpool\\n        self.dropout = baseline.classifier[0]\\n        self.heads = nn.ModuleDict({\\"eyepacs\\": baseline.classifier[1],\\n                                   \\"acrima\\": nn.Linear(1280, 2), \\"papila\\": nn.Linear(1280, 2)})\\n        for parameter in self.features[:7].parameters():\\n            parameter.requires_grad = False\\n        for parameter in self.features[7:].parameters():\\n            parameter.requires_grad = True\\n        self.train(False)\\n\\n    def train(self, mode=True):\\n        super().train(mode)\\n        self.features[:7].eval()\\n        for module in self.modules():\\n            if isinstance(module, nn.modules.batchnorm._BatchNorm):\\n                module.eval()\\n        return self\\n\\n    def prefix(self, images):\\n        return self.features[:7](images)\\n\\n    def embedding(self, prefix):\\n        return self.avgpool(self.features[7:](prefix)).flatten(1)\\n\\n    def task_logits(self, embedding, task):\\n        if task not in TASKS:\\n            raise ValueError(\\"Select an explicit research task, not a generic clinical output.\\")\\n        return self.heads[task](self.dropout(embedding))\\n\\n    def forward(self, images, task):\\n        return self.task_logits(self.embedding(self.prefix(images)), task)\\n\\n    def prefix_sha256(self):\\n        digest = sha256()\\n        for name, tensor in self.features[:7].state_dict().items():\\n            digest.update(name.encode())\\n            digest.update(str(tuple(tensor.shape)).encode())\\n            digest.update(tensor.detach().cpu().contiguous().numpy().tobytes())\\n        return digest.hexdigest()\\n\\n\\ndef load_separate_tasks(path):\\n    payload = torch.load(path, map_location=\\"cpu\\", weights_only=True)\\n    if payload.get(\\"architecture\\") != ARCHITECTURE or payload.get(\\"tasks\\") != list(TASKS):\\n        raise ValueError(\\"This is not a compatible separate-task research checkpoint.\\")\\n    if payload.get(\\"task_classes\\") != TASK_CLASSES or payload.get(\\"preprocessing\\") != PREPROCESSING:\\n        raise ValueError(\\"Task labels or preprocessing differ from the separate-task protocol.\\")\\n    trained = payload.get(\\"trained_tasks\\", [])\\n    thresholds = payload.get(\\"thresholds\\", {})\\n    if not trained or trained != [task for task in TASKS if task in trained] or set(thresholds) != set(trained):\\n        raise ValueError(\\"Trained tasks and thresholds must match explicitly.\\")\\n    if any(not isinstance(value, (int, float)) or not 0 < value < 1 for value in thresholds.values()):\\n        raise ValueError(\\"Task thresholds must be finite and between zero and one.\\")\\n    model = SeparateTaskModel()\\n    model.load_state_dict(payload[\\"state_dict\\"], strict=True)\\n    if any(not torch.isfinite(value).all() for value in model.state_dict().values()):\\n        raise ValueError(\\"Nonfinite checkpoint values.\\")\\n    if model.prefix_sha256() != payload[\\"frozen_prefix_sha256\\"]:\\n        raise ValueError(\\"Frozen-prefix provenance mismatch.\\")\\n    return model, {key: value for key, value in payload.items() if key != \\"state_dict\\"}\\n", "glaucoma/notebook_ui.py": "\\"\\"\\"An in-notebook judging dashboard. No tunnel, web server or disk uploads.\\"\\"\\"\\n\\nimport base64\\nfrom html import escape\\nfrom io import BytesIO\\n\\nfrom IPython.display import HTML, display\\nimport ipywidgets as widgets\\nimport torch\\n\\nfrom glaucoma.pipeline import InputError, ModelError, decode_image, infer, load_model, validate_image_signal\\n\\nSTYLE = \\"\\"\\"\\n<style>\\n.gai {font-family:Inter,Arial,sans-serif;color:#193e43;line-height:1.55;max-width:1100px;background:#fff;padding:18px;border-radius:16px}\\n.gai * {box-sizing:border-box}\\n.gai .hero {background:#113f40;color:#fff;padding:30px 34px;border-radius:18px;margin:0 0 20px}\\n.gai .eyebrow {font-size:11px;letter-spacing:2px;text-transform:uppercase;color:#a4ded0}\\n.gai h1 {font-size:32px;letter-spacing:-1px;margin:8px 0;color:#fff;font-weight:650}\\n.gai h2 {font-size:21px;margin:7px 0 12px;color:#193e43}\\n.gai p {margin:7px 0}\\n.gai .sub {color:#c9dfdc;font-size:14px}\\n.gai .grid {display:grid;grid-template-columns:1fr 1fr;gap:20px;margin:16px 0}\\n.gai .card {background:#f6f9fa;border:1px solid #dce8e6;border-radius:14px;padding:24px}\\n.gai .image {background:#101e26;border-radius:10px;min-height:260px;display:flex;align-items:center;justify-content:center;overflow:hidden}\\n.gai img {display:block;max-height:340px;max-width:100%;object-fit:contain}\\n.gai .label {font-size:11px;letter-spacing:1.4px;text-transform:uppercase;color:#607c80}\\n.gai .score {font-size:48px;font-weight:650;letter-spacing:-2px;color:#147d70}\\n.gai .track {height:14px;border-radius:20px;background:#e0ebe9;overflow:hidden;margin:10px 0}\\n.gai .fill {height:100%;border-radius:20px;background:#147d70}\\n.gai .row {display:flex;justify-content:space-between;gap:16px;font-size:13px;margin:8px 0}\\n.gai .note {border-left:3px solid #c99341;padding:12px 16px;background:#fff8ec;color:#6e542d;font-size:12px;border-radius:4px;margin:16px 0}\\n.gai .muted {font-size:12px;color:#607c80}\\n.gai .pill {display:inline-block;padding:4px 10px;background:#e2efeb;border-radius:20px;color:#266f62;font-size:11px}\\n@media(max-width:700px){.gai .grid{grid-template-columns:1fr}.gai .hero{padding:22px}.gai h1{font-size:27px}}\\n</style>\\n\\"\\"\\"\\n\\nDISCLAIMER = \\"\\"\\"<div class=\\"note\\"><b>Research prototype \\u00b7 not a diagnosis.</b>\\nThis model has not been clinically validated. Scores can be wrong and are not calibrated probabilities\\nof disease. Non-referable does not mean healthy. A qualified eye-care professional must interpret\\nretinal findings. The app does not check whether an upload is a fundus photograph.</div>\\"\\"\\"\\n\\n\\ndef uploaded_file(value):\\n    \\"\\"\\"Normalize both Colab\'s ipywidgets 7 mapping and ipywidgets 8 sequence.\\"\\"\\"\\n    if not value:\\n        return None\\n    if isinstance(value, dict):\\n        name, file = next(iter(value.items()))\\n        return name, bytes(file[\\"content\\"])\\n    file = value[0]\\n    return file[\\"name\\"], bytes(file[\\"content\\"])\\n\\n\\ndef image_data_url(image):\\n    preview = image.copy()\\n    preview.thumbnail((800, 800))\\n    buffer = BytesIO()\\n    preview.save(buffer, format=\\"PNG\\")\\n    return \\"data:image/png;base64,\\" + base64.b64encode(buffer.getvalue()).decode()\\n\\n\\ndef result_html(image, result):\\n    score = result[\\"referable\\"] * 100\\n    decision = \\"Above the referral-score threshold\\" if result[\\"flagged\\"] else \\"Below the referral-score threshold\\"\\n    return STYLE + f\\"\\"\\"<div class=\\"gai\\"><div class=\\"grid\\">\\n<div class=\\"card\\"><div class=\\"label\\">01 / Uploaded fundus image</div>\\n<div class=\\"image\\"><img src=\\"{image_data_url(image)}\\" alt=\\"Uploaded image for research analysis\\"/></div>\\n<p class=\\"muted\\">{image.width} \\u00d7 {image.height} original \\u00b7 224 \\u00d7 224 model input</p></div>\\n<div class=\\"card\\"><div class=\\"label\\">02 / Model output</div><h2>{decision}</h2>\\n<div class=\\"score\\">{score:.1f}<span style=\\"font-size:24px\\">%</span></div>\\n<p>Referable glaucoma model score</p>\\n<div class=\\"track\\" role=\\"progressbar\\" aria-label=\\"Referable glaucoma score\\" aria-valuemin=\\"0\\" aria-valuemax=\\"100\\" aria-valuenow=\\"{score:.1f}\\"><div class=\\"fill\\" style=\\"width:{score:.2f}%\\"></div></div>\\n<div class=\\"row\\"><span>Non-referable score</span><b>{result[\'non_referable\']:.1%}</b></div>\\n<div class=\\"row\\"><span>Referral-score threshold</span><b>{result[\'threshold\']:.1%}</b></div>\\n<div class=\\"row\\"><span>CPU preprocessing + inference</span><b>{result[\'elapsed_ms\']:.0f} ms</b></div>\\n<p class=\\"muted\\">The threshold was selected on validation images. This is a research routing signal, not a treatment recommendation.</p>\\n</div></div>{DISCLAIMER}</div>\\"\\"\\"\\n\\n\\ndef display_assistant(model_path=\\"models/best_model.pth\\"):\\n    \\"\\"\\"Call once in Colab/Jupyter after training; session memory only.\\"\\"\\"\\n    torch.set_num_threads(2)\\n    loaded = load_model(model_path)\\n    if loaded.metadata.get(\\"label_semantics\\") == \\"mixed_dataset_binary_research_labels\\":\\n        raise ModelError(\\"This experimental checkpoint mixes dataset labels. Use the sequential notebook\'s research results; it cannot be displayed as a referral score.\\")\\n    display(HTML(STYLE + f\\"\\"\\"<div class=\\"gai\\"><div class=\\"hero\\">\\n<div class=\\"eyebrow\\">GlaucomAI / AI + Health</div>\\n<h1>A clearer view. An earlier signal.</h1>\\n<p class=\\"sub\\">Explore glaucoma screening with a retinal photograph and a trained EfficientNet-B0.</p>\\n<span class=\\"pill\\">Trained checkpoint connected \\u00b7 CPU inference</span></div>\\n<p><b>Upload a fundus photograph</b> <span class=\\"muted\\">PNG or JPEG \\u00b7 under 10 MB \\u00b7 one image at a time</span></p>\\n{DISCLAIMER}</div>\\"\\"\\"))\\n    uploader = widgets.FileUpload(accept=\\".png,.jpg,.jpeg\\", multiple=False, description=\\"Choose image\\")\\n    analyze = widgets.Button(description=\\"Analyze image \\u2192\\", button_style=\\"success\\", disabled=True,\\n                             layout=widgets.Layout(width=\\"180px\\"))\\n    reset = widgets.Button(description=\\"Clear image\\", icon=\\"trash\\")\\n    output = widgets.Output()\\n    state = {\\"image\\": None}\\n\\n    def clear_image():\\n        if state[\\"image\\"] is not None:\\n            state[\\"image\\"].close()\\n        state[\\"image\\"] = None\\n        analyze.disabled = True\\n        output.clear_output()\\n\\n    def uploaded(change):\\n        clear_image()\\n        if not uploader.value:\\n            return\\n        try:\\n            name, content = uploaded_file(uploader.value)\\n            state[\\"image\\"] = decode_image(content, name)\\n            validate_image_signal(state[\\"image\\"])\\n            analyze.disabled = False\\n            with output:\\n                display(HTML(STYLE + f\'<div class=\\"gai\\"><p class=\\"pill\\">Image decoded \\u00b7 ready to analyze</p><div class=\\"image\\"><img src=\\"{image_data_url(state[\\"image\\"])}\\" alt=\\"Uploaded image preview\\"/></div></div>\'))\\n        except InputError as exc:\\n            clear_image()\\n            with output:\\n                display(HTML(f\'<div class=\\"gai\\"><div class=\\"note\\">{escape(str(exc))}</div></div>\'))\\n\\n    def run_analysis(button):\\n        if state[\\"image\\"] is None:\\n            return\\n        analyze.disabled = True\\n        analyze.description = \\"Analyzing\\u2026\\"\\n        output.clear_output(wait=True)\\n        try:\\n            result = infer(state[\\"image\\"], loaded)\\n            with output:\\n                display(HTML(result_html(state[\\"image\\"], result)))\\n        except (InputError, ModelError, RuntimeError) as exc:\\n            with output:\\n                display(HTML(f\'<div class=\\"gai\\"><div class=\\"note\\">Analysis unavailable: {escape(str(exc))}</div></div>\'))\\n        finally:\\n            analyze.description = \\"Analyze image \\u2192\\"\\n            analyze.disabled = state[\\"image\\"] is None\\n\\n    def reset_upload(button):\\n        clear_image()\\n        if isinstance(uploader.value, dict):\\n            # ipywidgets 7 computes its read-only value from these synced traits.\\n            uploader.metadata = []\\n            uploader.data = []\\n            uploader._counter = 0\\n            uploader.set_trait(\\"value\\", {})\\n        else:\\n            uploader.value = ()\\n\\n    uploader.observe(uploaded, names=\\"value\\")\\n    analyze.on_click(run_analysis)\\n    reset.on_click(reset_upload)\\n    display(widgets.VBox([widgets.HBox([uploader, analyze, reset]), output]))\\n    return {\\"uploader\\": uploader, \\"analyze\\": analyze, \\"reset\\": reset, \\"output\\": output}\\n", "glaucoma/pipeline.py": "\\"\\"\\"Shared, deterministic preprocessing and CPU inference. Never stores images.\\"\\"\\"\\n\\nfrom dataclasses import dataclass\\nfrom io import BytesIO\\nfrom pathlib import Path\\nimport math\\nimport time\\nimport warnings\\n\\nfrom PIL import Image, ImageOps, UnidentifiedImageError\\nimport torch\\nfrom torch import nn\\nfrom torchvision import models, transforms\\n\\nMAX_BYTES = 10 * 1024 * 1024\\nMAX_PIXELS = 25_000_000\\nCLASSES = [\\"non_referable\\", \\"referable\\"]\\nMEAN = [0.485, 0.456, 0.406]\\nSTD = [0.229, 0.224, 0.225]\\nPREPROCESSING = {\\"size\\": [224, 224], \\"mean\\": MEAN, \\"std\\": STD}\\n\\n\\nclass InputError(ValueError):\\n    pass\\n\\n\\nclass ModelError(ValueError):\\n    pass\\n\\n\\ndef decode_image(data: bytes, filename: str) -> Image.Image:\\n    \\"\\"\\"Check the actual format, byte size and pixel count, then remove metadata.\\"\\"\\"\\n    if Path(filename).suffix.lower() not in {\\".png\\", \\".jpg\\", \\".jpeg\\"}:\\n        raise InputError(\\"Choose a PNG or JPEG image.\\")\\n    if not data or len(data) >= MAX_BYTES:\\n        raise InputError(\\"Choose a non-empty image smaller than 10 MB.\\")\\n    try:\\n        with warnings.catch_warnings():\\n            warnings.simplefilter(\\"error\\", Image.DecompressionBombWarning)\\n            with Image.open(BytesIO(data)) as source:\\n                if source.format not in {\\"PNG\\", \\"JPEG\\"}:\\n                    raise InputError(\\"The file contents must be PNG or JPEG.\\")\\n                if source.width * source.height > MAX_PIXELS:\\n                    raise InputError(\\"The image exceeds the 25-megapixel safety limit.\\")\\n                source.verify()\\n            with Image.open(BytesIO(data)) as source:\\n                oriented = ImageOps.exif_transpose(source).convert(\\"RGB\\")\\n                # A new image avoids retaining EXIF, names or other source metadata.\\n                return Image.frombytes(\\"RGB\\", oriented.size, oriented.tobytes())\\n    except InputError:\\n        raise\\n    except (UnidentifiedImageError, OSError, SyntaxError, ValueError,\\n            Image.DecompressionBombError, Image.DecompressionBombWarning) as exc:\\n        raise InputError(\\"This image could not be decoded. Try another PNG or JPEG.\\") from exc\\n\\n\\ndef preprocessing():\\n    return transforms.Compose([\\n        transforms.Resize((224, 224), antialias=True),\\n        transforms.ToTensor(),\\n        transforms.Normalize(MEAN, STD),\\n    ])\\n\\n\\ndef validate_image_signal(image: Image.Image):\\n    \\"\\"\\"Reject an exactly uniform input; this is not a retinal-quality classifier.\\"\\"\\"\\n    extrema = image.convert(\\"RGB\\").getextrema()\\n    if all(low == high for low, high in extrema):\\n        raise InputError(\\"This image is completely uniform and contains no usable image detail. Choose another research image. No score was produced.\\")\\n\\n\\ndef build_model(pretrained: bool = False) -> nn.Module:\\n    weights = models.EfficientNet_B0_Weights.DEFAULT if pretrained else None\\n    model = models.efficientnet_b0(weights=weights)\\n    model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)\\n    return model\\n\\n\\n@dataclass\\nclass LoadedModel:\\n    model: nn.Module\\n    threshold: float\\n    metadata: dict\\n\\n\\ndef load_model(path: str | Path) -> LoadedModel:\\n    \\"\\"\\"Load only a local, trusted checkpoint. Never download inference weights.\\"\\"\\"\\n    try:\\n        checkpoint = torch.load(path, map_location=\\"cpu\\", weights_only=True)\\n        if not isinstance(checkpoint, dict):\\n            raise ModelError(\\"Checkpoint must be a dictionary exported by train.py.\\")\\n        if checkpoint.get(\\"architecture\\") != \\"efficientnet_b0\\":\\n            raise ModelError(\\"Checkpoint architecture must be efficientnet_b0.\\")\\n        if checkpoint.get(\\"classes\\") != CLASSES:\\n            raise ModelError(\\"Checkpoint class order must be non_referable=0, referable=1.\\")\\n        if checkpoint.get(\\"preprocessing\\") != PREPROCESSING:\\n            raise ModelError(\\"Checkpoint preprocessing does not match this app.\\")\\n        threshold = float(checkpoint[\\"threshold\\"])\\n        if not math.isfinite(threshold) or not 0 < threshold < 1:\\n            raise ModelError(\\"Checkpoint threshold must be between zero and one.\\")\\n        model = build_model()\\n        model.load_state_dict(checkpoint[\\"state_dict\\"], strict=True)\\n        model.eval()\\n        if any(not torch.isfinite(p).all() for p in model.state_dict().values()):\\n            raise ModelError(\\"Checkpoint contains invalid model values.\\")\\n        return LoadedModel(model, threshold, {\\n            key: value for key, value in checkpoint.items() if key != \\"state_dict\\"\\n        })\\n    except ModelError:\\n        raise\\n    except Exception as exc:\\n        raise ModelError(\\"Cannot load this checkpoint. Export a compatible model with train.py.\\") from exc\\n\\n\\ndef infer(image: Image.Image, loaded: LoadedModel) -> dict:\\n    start = time.perf_counter()\\n    validate_image_signal(image)\\n    with torch.inference_mode():\\n        tensor = preprocessing()(image.convert(\\"RGB\\")).unsqueeze(0)\\n        probabilities = torch.softmax(loaded.model(tensor), dim=1)[0]\\n    values = probabilities.tolist()\\n    if len(values) != 2 or not all(math.isfinite(p) for p in values):\\n        raise ModelError(\\"The model returned invalid scores. No result is available.\\")\\n    return {\\n        \\"non_referable\\": values[0],\\n        \\"referable\\": values[1],\\n        \\"threshold\\": loaded.threshold,\\n        \\"flagged\\": values[1] >= loaded.threshold,\\n        \\"elapsed_ms\\": (time.perf_counter() - start) * 1000,\\n        \\"demo\\": False,\\n    }\\n", "glaucoma/runtime.py": "\\"\\"\\"Stream child-process logs through notebook stdout and preserve failures.\\"\\"\\"\\n\\nimport subprocess\\n\\n\\ndef run_logged(arguments):\\n    with subprocess.Popen(arguments, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,\\n                          text=True, bufsize=1) as process:\\n        for line in process.stdout:\\n            print(line, end=\\"\\", flush=True)\\n        status = process.wait()\\n    if status:\\n        raise subprocess.CalledProcessError(status, arguments)\\n", "glaucoma/statistics.py": "\\"\\"\\"Research statistics with explicit uncertainty; no clinical release decision.\\"\\"\\"\\n\\nfrom collections import defaultdict\\nimport math\\nimport random\\n\\nfrom glaucoma.metrics import binary_metrics\\n\\n\\ndef wilson_interval(successes, total, z=1.959963984540054):\\n    \\"\\"\\"95% Wilson score interval; independence must be assessed separately.\\"\\"\\"\\n    if isinstance(total, bool) or isinstance(successes, bool) or not isinstance(total, int) or not isinstance(successes, int):\\n        raise ValueError(\\"Counts must be integers.\\")\\n    if not 0 <= successes <= total or total < 0:\\n        raise ValueError(\\"Invalid binomial counts.\\")\\n    if total == 0:\\n        return None\\n    p = successes / total\\n    denominator = 1 + z * z / total\\n    center = (p + z * z / (2 * total)) / denominator\\n    radius = z * math.sqrt(p * (1 - p) / total + z * z / (4 * total * total)) / denominator\\n    return [max(0., center - radius), min(1., center + radius)]\\n\\n\\ndef roc_auc(labels, scores):\\n    \\"\\"\\"Pair-ranking AUROC with half credit for ties.\\"\\"\\"\\n    binary_metrics(labels, scores)\\n    positives = sum(labels)\\n    negatives = len(labels) - positives\\n    if not positives or not negatives:\\n        return None\\n    pairs = sorted(zip(scores, labels))\\n    rank_sum, i = 0., 0\\n    while i < len(pairs):\\n        j = i + 1\\n        while j < len(pairs) and pairs[j][0] == pairs[i][0]:\\n            j += 1\\n        rank_sum += sum(label for _, label in pairs[i:j]) * ((i + 1 + j) / 2)\\n        i = j\\n    return (rank_sum - positives * (positives + 1) / 2) / (positives * negatives)\\n\\n\\ndef score_statistics(labels, scores, threshold):\\n    metrics = binary_metrics(labels, scores, threshold)\\n    tp, tn = metrics[\\"true_positive\\"], metrics[\\"true_negative\\"]\\n    fp, fn = metrics[\\"false_positive\\"], metrics[\\"false_negative\\"]\\n    metrics.update({\\"roc_auc\\": roc_auc(labels, scores),\\n                    \\"brier_score\\": sum((p - y) ** 2 for y, p in zip(labels, scores)) / len(labels),\\n                    \\"sample_ppv\\": tp / (tp + fp) if tp + fp else None,\\n                    \\"sample_npv\\": tn / (tn + fn) if tn + fn else None,\\n                    \\"sensitivity_wilson_95\\": wilson_interval(tp, tp + fn),\\n                    \\"specificity_wilson_95\\": wilson_interval(tn, tn + fp),\\n                    \\"interval_note\\": \\"Wilson intervals assume independent observations; eye-level intervals are descriptive when both eyes or repeated patients occur. PPV/NPV describe this sample, not population prevalence.\\"})\\n    return metrics\\n\\n\\ndef cluster_bootstrap(labels, scores, groups, threshold, repetitions=2000, seed=20261107):\\n    \\"\\"\\"Resample whole patients, preserving correlated eyes within each patient.\\"\\"\\"\\n    binary_metrics(labels, scores, threshold)\\n    if len(groups) != len(labels) or repetitions < 100:\\n        raise ValueError(\\"Provide one group per image and at least 100 replicates.\\")\\n    members = defaultdict(list)\\n    for index, group in enumerate(groups):\\n        members[group].append(index)\\n    keys = sorted(members)\\n    if len(keys) < 2:\\n        raise ValueError(\\"At least two groups are required.\\")\\n    rng = random.Random(seed)\\n    samples = {\\"sensitivity\\": [], \\"specificity\\": [], \\"accuracy\\": []}\\n    for _ in range(repetitions):\\n        indices = [i for key in rng.choices(keys, k=len(keys)) for i in members[key]]\\n        metrics = binary_metrics([labels[i] for i in indices], [scores[i] for i in indices], threshold)\\n        for key in samples:\\n            if metrics[key] is not None:\\n                samples[key].append(metrics[key])\\n\\n    def percentile(values, q):\\n        values = sorted(values)\\n        position = (len(values) - 1) * q\\n        lo, hi = math.floor(position), math.ceil(position)\\n        return values[lo] + (values[hi] - values[lo]) * (position - lo)\\n\\n    return {\\"groups\\": len(keys), \\"replicates\\": repetitions, \\"seed\\": seed,\\n            \\"method\\": \\"Percentile 95% bootstrap, sampling whole patient groups with replacement.\\",\\n            \\"intervals\\": {key: {\\"lower\\": percentile(values, .025), \\"upper\\": percentile(values, .975),\\n                                \\"valid_replicates\\": len(values)} if values else None\\n                          for key, values in samples.items()},\\n            \\"limitation\\": \\"Small cohorts and rare outcomes can produce unstable intervals. Grouping correctness, site independence, and sampling representativeness are not established by bootstrap.\\"}\\n", "input_guard_audit.py": "\\"\\"\\"Verify the exact-uniform input guard against real checkpoints and test images.\\"\\"\\"\\n\\nimport argparse\\nfrom collections import Counter\\nfrom hashlib import sha256\\nimport json\\nfrom pathlib import Path\\n\\nfrom PIL import Image\\n\\nfrom glaucoma.pipeline import InputError, decode_image, infer, load_model, validate_image_signal\\n\\n\\ndef main():\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument(\\"--run\\", type=Path, default=Path(\\"sequential_run\\"))\\n    args = parser.parse_args()\\n    source = json.loads((args.run / \\"results.json\\").read_text())\\n    rows = json.loads((args.run / \\"manifest.json\\").read_text())\\n    report = {\\"scope\\": \\"Exactly uniform RGB images only; not a validated retinal/gradable classifier.\\",\\n              \\"pipeline_source_sha256\\": sha256(Path(\\"glaucoma/pipeline.py\\").read_bytes()).hexdigest(),\\n              \\"controls\\": [], \\"test_images_checked\\": {}, \\"test_images_rejected\\": {},\\n              \\"clinical_release_allowed\\": False}\\n    checked, rejected = Counter(), Counter()\\n    for row in rows:\\n        if row[\\"split\\"] != \\"test\\":\\n            continue\\n        path = Path(row[\\"path\\"])\\n        checked[row[\\"dataset\\"]] += 1\\n        with decode_image(path.read_bytes(), path.name) as image:\\n            try:\\n                validate_image_signal(image)\\n            except InputError:\\n                rejected[row[\\"dataset\\"]] += 1\\n    report[\\"test_images_checked\\"] = dict(checked)\\n    report[\\"test_images_rejected\\"] = {key: rejected[key] for key in checked}\\n    for stage in source[\\"stages\\"]:\\n        path = args.run / stage[\\"checkpoint\\"]\\n        if sha256(path.read_bytes()).hexdigest() != stage[\\"checkpoint_sha256\\"]:\\n            raise ValueError(\\"Checkpoint differs from locked research run.\\")\\n        loaded = load_model(path)\\n        calls = []\\n        hook = loaded.model.register_forward_pre_hook(lambda *args: calls.append(True))\\n        for color in ((0, 0, 0), (255, 255, 255), (128, 128, 128), (220, 40, 20), (0, 180, 70)):\\n            try:\\n                infer(Image.new(\\"RGB\\", (512, 512), color), loaded)\\n            except InputError:\\n                report[\\"controls\\"].append({\\"stage\\": stage[\\"stage\\"], \\"color\\": list(color), \\"abstained\\": True})\\n            else:\\n                raise AssertionError(\\"A uniform control incorrectly produced a score.\\")\\n        hook.remove()\\n        if calls:\\n            raise AssertionError(\\"The model executed before input rejection.\\")\\n    report[\\"model_forward_calls_for_uniform_controls\\"] = 0\\n    (args.run / \\"input_guard_audit.json\\").write_text(json.dumps(report, indent=2))\\n    print(\\"INPUT GUARD AUDIT COMPLETE\\", flush=True)\\n    print(json.dumps(report, indent=2), flush=True)\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "multitask_train.py": "\\"\\"\\"Execute preregistered P2.0 in Colab; separate labels and tune suffix blocks.\\"\\"\\"\\n\\nimport argparse\\nfrom collections import Counter\\nfrom hashlib import sha256\\nimport json\\nfrom pathlib import Path\\nimport shutil\\n\\nimport torch\\nfrom torch.nn import functional as F\\nfrom torch.utils.data import DataLoader, Subset, TensorDataset\\n\\nfrom glaucoma.metrics import binary_metrics, select_threshold\\nfrom glaucoma.multidata import RecordDataset, replay_indices\\nfrom glaucoma.multitask import ARCHITECTURE, TASKS, TASK_CLASSES, SeparateTaskModel, load_separate_tasks\\nfrom glaucoma.pipeline import PREPROCESSING, load_model\\nfrom glaucoma.statistics import score_statistics\\n\\nSEED = 20261108\\n\\n\\ndef digest(path):\\n    return sha256(Path(path).read_bytes()).hexdigest()\\n\\n\\ndef cache_prefix(model, rows, device):\\n    model.eval()\\n    tensors, labels = [], []\\n    loader = DataLoader(RecordDataset(rows), batch_size=32)\\n    with torch.inference_mode():\\n        for i, (images, y) in enumerate(loader):\\n            tensors.append(model.prefix(images.to(device)).cpu())\\n            labels.append(y)\\n            if (i + 1) % 25 == 0 or i + 1 == len(loader):\\n                print(f\\"  Frozen prefix: {min((i + 1) * 32, len(rows))}/{len(rows)}\\", flush=True)\\n    return TensorDataset(torch.cat(tensors).clone(), torch.cat(labels).clone(),\\n                         torch.tensor([TASKS.index(row[\\"dataset\\"]) for row in rows]))\\n\\n\\ndef routed_loss(model, prefix, labels, tasks):\\n    embedding = model.embedding(prefix)\\n    loss = embedding.new_zeros(())\\n    for task_index, task in enumerate(TASKS):\\n        mask = tasks == task_index\\n        if mask.any():\\n            loss = loss + F.cross_entropy(model.task_logits(embedding[mask], task), labels[mask], reduction=\\"sum\\")\\n    return loss / len(labels)\\n\\n\\ndef predictions(model, cache, indices, task, device):\\n    model.eval()\\n    labels, scores, loss_sum = [], [], 0.\\n    with torch.inference_mode():\\n        for prefix, y, _ in DataLoader(Subset(cache, indices), batch_size=32):\\n            y = y.to(device)\\n            logits = model.task_logits(model.embedding(prefix.to(device)), task)\\n            loss_sum += F.cross_entropy(logits, y, reduction=\\"sum\\").item()\\n            labels.extend(y.cpu().tolist())\\n            scores.extend(logits.softmax(1)[:, 1].cpu().tolist())\\n    if not labels or set(labels) != {0, 1}:\\n        raise ValueError(\\"Every native-task validation/test population needs both classes.\\")\\n    return {\\"loss\\": loss_sum / len(labels), \\"labels\\": labels, \\"scores\\": scores}\\n\\n\\ndef validation(model, cache, rows, tasks, device):\\n    return {task: predictions(model, cache,\\n        [i for i, row in enumerate(rows) if row[\\"dataset\\"] == task and row[\\"split\\"] == \\"val\\"], task, device)\\n        for task in tasks}\\n\\n\\ndef train_stage(model, cache, rows, current, previous, parent, output, epochs, device, frozen_hash):\\n    active = previous + [current]\\n    indices = replay_indices(rows, current, previous)\\n    optimizer = torch.optim.AdamW([\\n        {\\"params\\": model.features[7:].parameters(), \\"lr\\": 1e-5},\\n        {\\"params\\": [p for task in active for p in model.heads[task].parameters()], \\"lr\\": 1e-3},\\n    ])\\n    loader = DataLoader(Subset(cache, indices), batch_size=32, shuffle=True)\\n    best_loss, best_state, best_epoch = float(\\"inf\\"), None, None\\n    history = []\\n    print(f\\"P2 STAGE {current}: {len(indices)} training/replay images; separate task labels\\", flush=True)\\n    for epoch in range(1, epochs + 1):\\n        model.train()\\n        total = 0.\\n        for prefix, labels, tasks in loader:\\n            optimizer.zero_grad()\\n            loss = routed_loss(model, prefix.to(device), labels.to(device), tasks.to(device))\\n            if not torch.isfinite(loss):\\n                raise ValueError(\\"Nonfinite training loss; stopping without promotion.\\")\\n            loss.backward()\\n            optimizer.step()\\n            total += loss.item() * len(labels)\\n        values = validation(model, cache, rows, active, device)\\n        val_loss = sum(value[\\"loss\\"] for value in values.values()) / len(values)\\n        if model.prefix_sha256() != frozen_hash:\\n            raise ValueError(\\"Frozen prefix changed; cached features are invalid.\\")\\n        history.append({\\"epoch\\": epoch, \\"training_loss\\": total / len(indices), \\"validation_macro_loss\\": val_loss,\\n                        \\"validation_task_loss\\": {key: value[\\"loss\\"] for key, value in values.items()}})\\n        print(f\\"  Epoch {epoch:02d}/{epochs}: train {total/len(indices):.4f}, native-task macro val {val_loss:.4f}\\", flush=True)\\n        if val_loss < best_loss:\\n            best_loss, best_epoch = val_loss, epoch\\n            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}\\n    if best_state is None or any(not torch.isfinite(value).all() for value in best_state.values()):\\n        raise ValueError(\\"No finite checkpoint available.\\")\\n    model.load_state_dict(best_state)\\n    values = validation(model, cache, rows, active, device)\\n    thresholds = {key: select_threshold(value[\\"labels\\"], value[\\"scores\\"]) for key, value in values.items()}\\n    metadata = {\\"architecture\\": ARCHITECTURE, \\"tasks\\": list(TASKS), \\"trained_tasks\\": active,\\n                \\"task_classes\\": TASK_CLASSES,\\n                \\"state_dict\\": best_state, \\"preprocessing\\": PREPROCESSING,\\n                \\"thresholds\\": thresholds, \\"seed\\": SEED, \\"epoch\\": best_epoch, \\"epochs_run\\": epochs,\\n                \\"frozen_prefix_sha256\\": frozen_hash, \\"parent_checkpoint_sha256\\": digest(parent),\\n                \\"training_domain_counts\\": dict(Counter(rows[i][\\"dataset\\"] for i in indices)),\\n                \\"protocol\\": \\"P2.0\\", \\"status\\": \\"research_only_unvalidated\\",\\n                \\"label_meanings\\": {\\"eyepacs\\": \\"dataset referral label\\", \\"acrima\\": \\"expert image label\\",\\n                                   \\"papila\\": \\"dataset clinical glaucoma label\\"},\\n                \\"validation_metrics\\": {key: binary_metrics(value[\\"labels\\"], value[\\"scores\\"], thresholds[key])\\n                                       for key, value in values.items()},\\n                \\"validation_macro_loss\\": best_loss}\\n    torch.save(metadata, output)\\n    output.with_suffix(\\".history.json\\").write_text(json.dumps(history, indent=2))\\n    result = {key: value for key, value in metadata.items() if key != \\"state_dict\\"}\\n    result.update({\\"checkpoint\\": output.name, \\"checkpoint_sha256\\": digest(output)})\\n    print(\\"SAVED\\", output.name, result[\\"checkpoint_sha256\\"], flush=True)\\n    return result\\n\\n\\ndef main():\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument(\\"--manifest\\", type=Path, default=Path(\\"sequential_run/manifest.json\\"))\\n    parser.add_argument(\\"--baseline\\", type=Path, default=Path(\\"sequential_run/01_eyepacs.pth\\"))\\n    parser.add_argument(\\"--work\\", type=Path, default=Path(\\"multitask_run\\"))\\n    parser.add_argument(\\"--epochs\\", type=int, default=12)\\n    args = parser.parse_args()\\n    if not torch.cuda.is_available():\\n        raise RuntimeError(\\"P2 requires a Colab GPU; no local/CPU fallback.\\")\\n    if args.epochs != 12:\\n        raise ValueError(\\"P2.0 specifies 12 epochs; revise the protocol before changing this.\\")\\n    if args.work.exists():\\n        raise ValueError(\\"Choose a new output folder; prior runs must remain intact.\\")\\n    torch.set_num_threads(2)\\n    torch.manual_seed(SEED)\\n    device = torch.device(\\"cuda\\")\\n    loaded = load_model(args.baseline)\\n    if loaded.metadata.get(\\"dataset_provenance\\", {}).get(\\"dataset\\") != \\"deathtrooper/glaucoma-dataset-eyepacs-airogs-light-v2/versions/4\\":\\n        raise ValueError(\\"Initialization must be the original EyePACS-only checkpoint.\\")\\n    rows = json.loads(args.manifest.read_text())\\n    original_run = json.loads((args.manifest.parent / \\"results.json\\").read_text())\\n    if digest(args.baseline) != original_run[\\"stages\\"][0][\\"checkpoint_sha256\\"]:\\n        raise ValueError(\\"Baseline is not the audited first-stage checkpoint.\\")\\n    counts = dict(Counter(f\\"{r[\'dataset\']}/{r[\'split\']}/{r[\'label\']}\\" for r in rows))\\n    if counts != original_run[\\"counts\\"]:\\n        raise ValueError(\\"Manifest counts differ from the audited initialization record.\\")\\n    seen_hashes, groups = set(), {}\\n    for row in rows:\\n        if row[\\"dataset\\"] not in TASKS or row[\\"label\\"] not in (0, 1) or row[\\"split\\"] not in (\\"train\\", \\"val\\", \\"test\\"):\\n            raise ValueError(\\"Manifest has unsupported task/label/split.\\")\\n        if digest(row[\\"path\\"]) != row[\\"sha256\\"]:\\n            raise ValueError(\\"An input changed since the audited S1 manifest.\\")\\n        if row[\\"sha256\\"] in seen_hashes:\\n            raise ValueError(\\"Duplicate image in the manifest.\\")\\n        seen_hashes.add(row[\\"sha256\\"])\\n        group = (row[\\"dataset\\"], row[\\"group\\"])\\n        if group in groups and groups[group] != row[\\"split\\"]:\\n            raise ValueError(\\"Patient/group crosses splits.\\")\\n        groups[group] = row[\\"split\\"]\\n    args.work.mkdir(parents=True)\\n    shutil.copyfile(args.manifest, args.work / \\"manifest.json\\")\\n    model = SeparateTaskModel(loaded.model).to(device)\\n    frozen_hash = model.prefix_sha256()\\n    print(\\"P2 TRAINING IN COLAB\\", torch.cuda.get_device_name(0), \\"frozen prefix\\", frozen_hash, flush=True)\\n    feature_rows = [row for row in rows if row[\\"split\\"] != \\"test\\"]\\n    cache = cache_prefix(model, feature_rows, device)\\n    stages, parent = [], args.baseline\\n    for index, current in enumerate((\\"acrima\\", \\"papila\\"), 2):\\n        output = args.work / f\\"{index:02d}_{current}_separate_tasks.pth\\"\\n        previous = [\\"eyepacs\\"] + ([\\"acrima\\"] if current == \\"papila\\" else [])\\n        record = train_stage(model, cache, feature_rows, current, previous, parent, output,\\n                             args.epochs, device, frozen_hash)\\n        stages.append(record)\\n        parent = output\\n        (args.work / \\"stages.json\\").write_text(json.dumps(stages, indent=2))\\n    print(\\"P2 CHECKPOINTS LOCKED. Existing test partitions are inspected development benchmarks.\\", flush=True)\\n    test_rows = [row for row in rows if row[\\"split\\"] == \\"test\\"]\\n    test_cache = cache_prefix(model, test_rows, device)\\n    for stage in stages:\\n        candidate, metadata = load_separate_tasks(args.work / stage[\\"checkpoint\\"])\\n        candidate = candidate.to(device)\\n        stage[\\"development_benchmark_metrics\\"] = {}\\n        for task in metadata[\\"trained_tasks\\"]:\\n            values = predictions(candidate, test_cache,\\n                [i for i, row in enumerate(test_rows) if row[\\"dataset\\"] == task], task, device)\\n            stage[\\"development_benchmark_metrics\\"][task] = score_statistics(values[\\"labels\\"], values[\\"scores\\"], metadata[\\"thresholds\\"][task])\\n    report = {\\"protocol\\": \\"P2.0\\", \\"device\\": torch.cuda.get_device_name(0), \\"seed\\": SEED,\\n              \\"source_run_sha256\\": digest(args.manifest.parent / \\"results.json\\"),\\n              \\"initialization_sha256\\": digest(args.baseline), \\"manifest_sha256\\": digest(args.manifest),\\n              \\"frozen_prefix_sha256\\": frozen_hash, \\"stages\\": stages, \\"clinical_release_allowed\\": False,\\n              \\"test_status\\": \\"Previously inspected development benchmarks; not independent clinical confirmation.\\",\\n              \\"scope\\": \\"Three separate source-specific targets; no generic clinical output or automated source selection.\\"}\\n    report[\\"source_sha256\\"] = {name: digest(name) for name in\\n        (\\"multitask_train.py\\", \\"glaucoma/multitask.py\\", \\"glaucoma/multidata.py\\", \\"glaucoma/pipeline.py\\")}\\n    (args.work / \\"results.json\\").write_text(json.dumps(report, indent=2))\\n    print(\\"P2 RESEARCH TRAINING COMPLETE\\", flush=True)\\n    print(json.dumps(report, indent=2), flush=True)\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "prepare_data.py": "\\"\\"\\"Download the pinned Kaggle dataset and preserve its supplied splits.\\"\\"\\"\\n\\nimport argparse\\nfrom collections import Counter\\nimport csv\\nfrom hashlib import sha256\\nimport json\\nfrom pathlib import Path\\nimport shutil\\n\\nfrom glaucoma.data import FundusDataset, check_split_overlap\\nfrom glaucoma.pipeline import decode_image\\n\\nDATASET = \\"deathtrooper/glaucoma-dataset-eyepacs-airogs-light-v2/versions/4\\"\\nSOURCE = \\"https://www.kaggle.com/datasets/deathtrooper/glaucoma-dataset-eyepacs-airogs-light-v2/versions/4\\"\\nSPLITS = {\\"train\\": \\"train\\", \\"validation\\": \\"val\\", \\"test\\": \\"test\\"}\\nLABELS = {\\"NRG\\": \\"non_referable\\", \\"RG\\": \\"referable\\"}\\n\\n\\ndef prepare(source: Path, output: Path):\\n    \\"\\"\\"Validate every file before copying; no guessed labels or random re-split.\\"\\"\\"\\n    roots = [p for p in source.rglob(\\"*\\") if p.is_dir()\\n             and all((p / name).is_dir() for name in SPLITS)]\\n    if all((source / name).is_dir() for name in SPLITS):\\n        roots.insert(0, source)\\n    if len(roots) != 1:\\n        raise ValueError(\\"Expected one dataset root with train/, validation/ and test/ folders.\\")\\n    root = roots[0]\\n    if output.exists():\\n        raise ValueError(f\\"{output} already exists. Choose a new output directory to avoid mixing splits.\\")\\n    rows = []\\n    seen = {}\\n    for source_split, split in SPLITS.items():\\n        for source_class, label in LABELS.items():\\n            folder = root / source_split / source_class\\n            paths = sorted(p for p in folder.rglob(\\"*\\")\\n                           if p.is_file() and p.suffix.lower() in {\\".jpg\\", \\".jpeg\\", \\".png\\"})\\n            if not paths:\\n                raise ValueError(f\\"No images found in {folder}.\\")\\n            for path in paths:\\n                raw = path.read_bytes()\\n                decode_image(raw, path.name).close()\\n                digest = sha256(raw).hexdigest()\\n                if digest in seen:\\n                    raise ValueError(f\\"Duplicate image: {path} and {seen[digest]}. Review source data.\\")\\n                seen[digest] = path\\n                rows.append({\\n                    \\"source\\": path.relative_to(root).as_posix(), \\"split\\": split,\\n                    \\"label\\": label, \\"sha256\\": digest,\\n                    \\"destination\\": f\\"{split}/{label}/{digest[:16]}{path.suffix.lower()}\\",\\n                })\\n    # Validation succeeds before creating the output directory.\\n    output.mkdir(parents=True)\\n    for row in rows:\\n        destination = output / row[\\"destination\\"]\\n        destination.parent.mkdir(parents=True, exist_ok=True)\\n        shutil.copyfile(root / row[\\"source\\"], destination)\\n    with (output / \\"manifest.csv\\").open(\\"w\\", newline=\\"\\") as stream:\\n        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))\\n        writer.writeheader()\\n        writer.writerows(rows)\\n    check_split_overlap(*[FundusDataset(output / split) for split in SPLITS.values()])\\n    provenance = {\\n        \\"dataset\\": DATASET, \\"source_url\\": SOURCE,\\n        \\"counts\\": dict(Counter(f\\"{r[\'split\']}/{r[\'label\']}\\" for r in rows)),\\n        \\"split_policy\\": \\"Preserve the dataset publisher\'s train/validation/test splits.\\",\\n        \\"patient_separation\\": \\"Not independently verified. File-hash disjointness does not establish patient disjointness.\\",\\n        \\"label_mapping\\": LABELS,\\n    }\\n    (output / \\"provenance.json\\").write_text(json.dumps(provenance, indent=2))\\n    return provenance\\n\\n\\ndef main():\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument(\\"--source\\", type=Path, help=\\"Use an already downloaded Kaggle dataset root.\\")\\n    parser.add_argument(\\"--output\\", type=Path, default=Path(\\"data/prepared\\"))\\n    args = parser.parse_args()\\n    if args.source:\\n        source = args.source\\n    else:\\n        import kagglehub\\n        source = Path(kagglehub.dataset_download(DATASET))\\n    print(json.dumps(prepare(source, args.output), indent=2))\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "sequential_train.py": "\\"\\"\\"Colab-only sequential adaptation: EyePACS -> ACRIMA -> PAPILA, with replay.\\"\\"\\"\\n\\nimport argparse\\nfrom collections import Counter\\nfrom hashlib import sha256\\nimport json\\nfrom pathlib import Path\\nimport shutil\\nimport subprocess\\nimport sys\\nimport zipfile\\n\\nimport torch\\nfrom torch import nn\\nfrom torch.utils.data import DataLoader, Subset\\n\\nfrom glaucoma.metrics import binary_metrics\\nfrom glaucoma.multidata import (HANDLES, SEED, RecordDataset, acrima_rows, audit_rows,\\n                               eyepacs_rows, papila_rows, replay_indices)\\nfrom glaucoma.pipeline import load_model\\nfrom glaucoma.runtime import run_logged\\nfrom train import extract_features\\n\\n\\ndef digest(path):\\n    return sha256(Path(path).read_bytes()).hexdigest()\\n\\n\\ndef validation(model, features, rows, domains, device):\\n    model.eval()\\n    result = {}\\n    with torch.inference_mode():\\n        for domain in domains:\\n            indices = [i for i, r in enumerate(rows) if r[\\"dataset\\"] == domain and r[\\"split\\"] == \\"val\\"]\\n            x, y = (v[indices].to(device) for v in features.tensors)\\n            logits = model.classifier(x)\\n            result[domain] = {\\"loss\\": nn.functional.cross_entropy(logits, y).item(),\\n                              \\"labels\\": y.cpu().tolist(), \\"scores\\": logits.softmax(1)[:, 1].cpu().tolist()}\\n    return result\\n\\n\\ndef domain_threshold(predictions, target=.88):\\n    \\"\\"\\"Give each source equal weight when selecting the validation threshold.\\"\\"\\"\\n    candidates = sorted({.5, *(p for v in predictions.values() for p in v[\\"scores\\"] if 0 < p < 1)})\\n    best, best_specificity = None, -1.\\n    for threshold in candidates:\\n        metrics = [binary_metrics(v[\\"labels\\"], v[\\"scores\\"], threshold) for v in predictions.values()]\\n        recall = sum(m[\\"sensitivity\\"] for m in metrics) / len(metrics)\\n        specificity = sum(m[\\"specificity\\"] for m in metrics) / len(metrics)\\n        if recall >= target and specificity > best_specificity:\\n            best, best_specificity = threshold, specificity\\n    if best is None:\\n        raise ValueError(\\"No valid validation threshold reaches the requested macro sensitivity.\\")\\n    return best\\n\\n\\ndef adapt(parent, output, features, rows, domain, previous, epochs, device):\\n    torch.manual_seed(SEED)\\n    loaded = load_model(parent)\\n    model = loaded.model.to(device)\\n    for parameter in model.features.parameters():\\n        parameter.requires_grad = False\\n    indices = replay_indices(rows, domain, previous)\\n    loader = DataLoader(Subset(features, indices), batch_size=32, shuffle=True)\\n    domains = previous + [domain]\\n    optimizer = torch.optim.AdamW(model.classifier.parameters(), lr=3e-4)\\n    best_loss, best_state, best_epoch = float(\\"inf\\"), None, None\\n    history = []\\n    print(f\\"STAGE {domain}: continue {digest(parent)}; {len(indices)} new/replay training examples\\", flush=True)\\n    for epoch in range(1, epochs + 1):\\n        model.classifier.train()\\n        loss_sum = 0.\\n        for x, y in loader:\\n            x, y = x.to(device), y.to(device)\\n            optimizer.zero_grad()\\n            loss = nn.functional.cross_entropy(model.classifier(x), y)\\n            loss.backward()\\n            optimizer.step()\\n            loss_sum += loss.item() * len(y)\\n        values = validation(model, features, rows, domains, device)\\n        val_loss = sum(v[\\"loss\\"] for v in values.values()) / len(values)\\n        history.append({\\"epoch\\": epoch, \\"train_loss\\": loss_sum / len(indices),\\n                        \\"validation_macro_loss\\": val_loss,\\n                        \\"validation_domain_loss\\": {k: v[\\"loss\\"] for k, v in values.items()}})\\n        print(f\\"  Epoch {epoch:02d}/{epochs}: train={loss_sum / len(indices):.4f}, macro val={val_loss:.4f}\\", flush=True)\\n        if val_loss < best_loss:\\n            best_loss, best_epoch = val_loss, epoch\\n            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}\\n    model.load_state_dict(best_state)\\n    values = validation(model, features, rows, domains, device)\\n    threshold = domain_threshold(values)\\n    metadata = dict(loaded.metadata)\\n    metadata.update({\\"state_dict\\": best_state, \\"threshold\\": threshold, \\"epoch\\": best_epoch,\\n                     \\"seed\\": SEED, \\"parent_checkpoint_sha256\\": digest(parent),\\n                     \\"stage\\": domain, \\"training_domains\\": domains,\\n                     \\"training_samples\\": len(indices),\\n                     \\"training_domain_counts\\": dict(Counter(rows[i][\\"dataset\\"] for i in indices)),\\n                     \\"dataset_provenance\\": {k: HANDLES[k] for k in domains},\\n                     \\"label_semantics\\": \\"mixed_dataset_binary_research_labels\\",\\n                     \\"label_note\\": \\"EyePACS referral labels differ from ACRIMA/PAPILA glaucoma labels; scores are not clinical referral risks.\\",\\n                     \\"validation_loss\\": best_loss,\\n                     \\"validation_metrics\\": {k: binary_metrics(v[\\"labels\\"], v[\\"scores\\"], threshold) for k, v in values.items()},\\n                     \\"status\\": \\"research_only_unvalidated\\"})\\n    torch.save(metadata, output)\\n    output.with_suffix(\\".history.json\\").write_text(json.dumps(history, indent=2))\\n    record = {k: v for k, v in metadata.items() if k != \\"state_dict\\"}\\n    record.update({\\"checkpoint\\": output.name, \\"checkpoint_sha256\\": digest(output), \\"epochs_run\\": epochs})\\n    print(f\\"SAVED {output.name} sha256={record[\'checkpoint_sha256\']}\\", flush=True)\\n    return record\\n\\n\\ndef main():\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument(\\"--work\\", type=Path, default=Path(\\"sequential_run\\"))\\n    parser.add_argument(\\"--baseline\\", type=Path, help=\\"Reuse a trusted EyePACS-only checkpoint trained in this Colab runtime.\\")\\n    parser.add_argument(\\"--eyepacs-prepared\\", type=Path, default=Path(\\"data/prepared\\"))\\n    parser.add_argument(\\"--epochs\\", type=int, default=12)\\n    args = parser.parse_args()\\n    if args.epochs < 1:\\n        parser.error(\\"epochs must be positive\\")\\n    if not torch.cuda.is_available():\\n        raise RuntimeError(\\"This workflow requires a Colab GPU; choose a T4 runtime before running.\\")\\n    if args.work.exists():\\n        raise ValueError(\\"Output directory already exists; choose a new --work to preserve prior runs.\\")\\n    torch.set_num_threads(2)\\n    torch.manual_seed(SEED)\\n    device = torch.device(\\"cuda\\")\\n    print(\\"TRAINING IN COLAB:\\", torch.cuda.get_device_name(0), flush=True)\\n    import kagglehub\\n    from prepare_data import prepare\\n\\n    if not args.eyepacs_prepared.exists():\\n        prepare(Path(kagglehub.dataset_download(HANDLES[\\"eyepacs\\"])), args.eyepacs_prepared)\\n    rows = eyepacs_rows(args.eyepacs_prepared)\\n    for domain, adapter in ((\\"acrima\\", acrima_rows), (\\"papila\\", papila_rows)):\\n        source = Path(kagglehub.dataset_download(HANDLES[domain]))\\n        rows.extend(adapter(source))\\n    counts = audit_rows(rows)\\n    print(\\"LOCKED SPLITS\\", json.dumps(counts, sort_keys=True), flush=True)\\n    args.work.mkdir(parents=True)\\n    (args.work / \\"manifest.json\\").write_text(json.dumps(rows, indent=2))\\n    baseline = args.work / \\"01_eyepacs.pth\\"\\n    if args.baseline:\\n        source = load_model(args.baseline)\\n        if source.metadata.get(\\"label_semantics\\") or source.metadata.get(\\"training_domains\\"):\\n            raise ValueError(\\"Baseline must be EyePACS-only, not an adapted checkpoint.\\")\\n        provenance = source.metadata.get(\\"dataset_provenance\\", {})\\n        if provenance.get(\\"dataset\\") != HANDLES[\\"eyepacs\\"] or source.metadata.get(\\"training_samples\\") != 8000:\\n            raise ValueError(\\"Baseline provenance must identify the pinned 8,000-image EyePACS training split.\\")\\n        shutil.copyfile(args.baseline, baseline)\\n    else:\\n        run_logged([sys.executable, \\"-u\\", \\"train.py\\", \\"--data\\", str(args.eyepacs_prepared),\\n                    \\"--epochs\\", str(args.epochs), \\"--batch-size\\", \\"32\\", \\"--device\\", \\"cuda\\", \\"--output\\", str(baseline)])\\n    loaded = load_model(baseline)\\n    record = {k: v for k, v in loaded.metadata.items()}\\n    record.update({\\"stage\\": \\"eyepacs\\", \\"checkpoint\\": baseline.name, \\"checkpoint_sha256\\": digest(baseline),\\n                   \\"reused_existing_colab_stage\\": bool(args.baseline)})\\n    stages = [record]\\n    feature_rows = [r for r in rows if r[\\"split\\"] != \\"test\\"]\\n    print(\\"Caching frozen features for training and validation only.\\", flush=True)\\n    features = extract_features(loaded.model.to(device), RecordDataset(feature_rows), device, 32)\\n    parent = baseline\\n    for index, domain in enumerate((\\"acrima\\", \\"papila\\"), 2):\\n        output = args.work / f\\"{index:02d}_{domain}.pth\\"\\n        stages.append(adapt(parent, output, features, feature_rows, domain,\\n                            [\\"eyepacs\\"] + ([\\"acrima\\"] if domain == \\"papila\\" else []), args.epochs, device))\\n        parent = output\\n        (args.work / \\"stages.json\\").write_text(json.dumps(stages, indent=2))\\n    # Test extraction and every test evaluation happen only after all stages are locked.\\n    print(\\"ALL STAGES LOCKED. Evaluating withheld test images once per checkpoint.\\", flush=True)\\n    test_rows = [r for r in rows if r[\\"split\\"] == \\"test\\"]\\n    test_features = extract_features(load_model(parent).model.to(device), RecordDataset(test_rows), device, 32)\\n    for stage in stages:\\n        model = load_model(args.work / stage[\\"checkpoint\\"]).model.to(device).eval()\\n        with torch.inference_mode():\\n            scores = model.classifier(test_features.tensors[0].to(device)).softmax(1)[:, 1].cpu().tolist()\\n        stage[\\"test_metrics\\"] = {}\\n        for domain in HANDLES:\\n            indices = [i for i, r in enumerate(test_rows) if r[\\"dataset\\"] == domain]\\n            stage[\\"test_metrics\\"][domain] = binary_metrics(\\n                [test_rows[i][\\"label\\"] for i in indices], [scores[i] for i in indices], stage[\\"threshold\\"])\\n    report = {\\"training_device\\": torch.cuda.get_device_name(0), \\"seed\\": SEED,\\n              \\"datasets\\": HANDLES, \\"counts\\": counts, \\"stages\\": stages,\\n              \\"label_semantics\\": \\"Dataset-specific binary labels; not interchangeable clinical diagnoses.\\",\\n              \\"test_policy\\": \\"All checkpoints locked before test evaluation. EyePACS test was inspected in the earlier baseline run; new ACRIMA/PAPILA test splits were withheld throughout adaptation.\\",\\n              \\"split_limitations\\": \\"PAPILA patient-group split; ACRIMA image split (patient IDs unavailable); EyePACS publisher splits, patient separation unverified. Byte/pixel hashes reject exact duplicates, not all near-duplicates.\\",\\n              \\"method\\": \\"Frozen ImageNet EfficientNet-B0; sequential head adaptation with bounded replay. Macro source validation loss chooses epochs; equal-source validation sensitivity >=88% chooses thresholds.\\",\\n              \\"status\\": \\"research_only_unvalidated\\"}\\n    (args.work / \\"results.json\\").write_text(json.dumps(report, indent=2))\\n    versions = subprocess.run([sys.executable, \\"-m\\", \\"pip\\", \\"freeze\\"], check=True, capture_output=True, text=True).stdout\\n    (args.work / \\"environment.txt\\").write_text(versions)\\n    archive = args.work.with_suffix(\\".zip\\")\\n    with zipfile.ZipFile(archive, \\"w\\", zipfile.ZIP_DEFLATED) as bundle:\\n        for path in args.work.iterdir():\\n            # The full image manifest stays private to the runtime; no images or clinical tables are exported.\\n            if path.name != \\"manifest.json\\":\\n                bundle.write(path, arcname=path.name)\\n    print(\\"SEQUENTIAL TRAINING COMPLETE\\", flush=True)\\n    print(json.dumps(report, indent=2), flush=True)\\n    print(\\"Archive:\\", archive, flush=True)\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "train.py": "\\"\\"\\"Train a frozen EfficientNet-B0 backbone, select threshold on validation only.\\"\\"\\"\\n\\nimport argparse\\nimport json\\nfrom pathlib import Path\\nimport random\\n\\nimport torch\\nfrom torch import nn\\nfrom torch.utils.data import DataLoader, TensorDataset\\n\\nfrom glaucoma.data import FundusDataset, check_split_overlap\\nfrom glaucoma.metrics import binary_metrics, select_threshold\\nfrom glaucoma.pipeline import CLASSES, PREPROCESSING, build_model\\n\\n\\ndef extract_features(model, dataset, device, batch_size):\\n    \\"\\"\\"A frozen backbone with deterministic preprocessing only needs one pass.\\"\\"\\"\\n    model.eval()\\n    features, targets = [], []\\n    loader = DataLoader(dataset, batch_size=batch_size)\\n    with torch.inference_mode():\\n        for index, (images, labels) in enumerate(loader):\\n            output = model.features(images.to(device))\\n            output = model.avgpool(output).flatten(1)\\n            features.append(output.cpu())\\n            targets.append(labels)\\n            if (index + 1) % 25 == 0 or index + 1 == len(loader):\\n                print(f\\"  Backbone features: {min((index + 1) * batch_size, len(dataset))}/{len(dataset)}\\", flush=True)\\n    # Clone outside inference mode so the classifier can save inputs for backward.\\n    return TensorDataset(torch.cat(features).clone(), torch.cat(targets).clone())\\n\\n\\ndef main():\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument(\\"--data\\", type=Path, default=Path(\\"data/prepared\\"))\\n    parser.add_argument(\\"--output\\", type=Path, default=Path(\\"models/best_model.pth\\"))\\n    parser.add_argument(\\"--epochs\\", type=int, default=12)\\n    parser.add_argument(\\"--batch-size\\", type=int, default=16)\\n    parser.add_argument(\\"--seed\\", type=int, default=42)\\n    parser.add_argument(\\"--device\\", choices=[\\"auto\\", \\"cpu\\", \\"cuda\\", \\"mps\\"], default=\\"auto\\")\\n    args = parser.parse_args()\\n    if args.epochs < 1 or args.batch_size < 1:\\n        parser.error(\\"epochs and batch-size must be positive\\")\\n    torch.manual_seed(args.seed)\\n    random.seed(args.seed)\\n    torch.set_num_threads(2)\\n    train = FundusDataset(args.data / \\"train\\")\\n    val = FundusDataset(args.data / \\"val\\")\\n    check_split_overlap(train, val)\\n    device = torch.device((\\"cuda\\" if torch.cuda.is_available() else \\"cpu\\")\\n                          if args.device == \\"auto\\" else args.device)\\n    print(f\\"Training on {device}; {len(train)} train / {len(val)} validation images.\\")\\n    print(\\"ImageNet weights download on first training run. No dataset is downloaded.\\")\\n    model = build_model(pretrained=True).to(device)\\n    for p in model.features.parameters():\\n        p.requires_grad = False\\n    optimizer = torch.optim.AdamW(model.classifier.parameters(), lr=1e-3)\\n    criterion = nn.CrossEntropyLoss()\\n    print(\\"Extracting frozen backbone features once for train and validation.\\", flush=True)\\n    train_features = extract_features(model, train, device, args.batch_size)\\n    val_features = extract_features(model, val, device, args.batch_size)\\n    loader = DataLoader(train_features, batch_size=args.batch_size, shuffle=True)\\n    val_loader = DataLoader(val_features, batch_size=args.batch_size)\\n    best_loss = float(\\"inf\\")\\n    history = []\\n    args.output.parent.mkdir(parents=True, exist_ok=True)\\n    for epoch in range(args.epochs):\\n        model.train()\\n        model.features.eval()  # Also freeze batchnorm statistics in the backbone.\\n        train_loss = 0.0\\n        for features, labels in loader:\\n            features, labels = features.to(device), labels.to(device)\\n            optimizer.zero_grad()\\n            loss = criterion(model.classifier(features), labels)\\n            loss.backward()\\n            optimizer.step()\\n            train_loss += loss.item() * len(labels)\\n        model.eval()\\n        val_loss, all_labels, probabilities = 0.0, [], []\\n        with torch.inference_mode():\\n            for features, labels in val_loader:\\n                logits = model.classifier(features.to(device))\\n                val_loss += criterion(logits, labels.to(device)).item() * len(labels)\\n                all_labels.extend(labels.tolist())\\n                probabilities.extend(logits.softmax(dim=1)[:, 1].cpu().tolist())\\n        val_loss /= len(val)\\n        print(f\\"Epoch {epoch + 1:02d}: train loss {train_loss / len(train):.4f}, val loss {val_loss:.4f}\\")\\n        history.append({\\"epoch\\": epoch + 1, \\"train_loss\\": train_loss / len(train), \\"validation_loss\\": val_loss})\\n        args.output.with_suffix(\\".history.json\\").write_text(json.dumps(history, indent=2))\\n        if val_loss < best_loss:\\n            best_loss = val_loss\\n            threshold = select_threshold(all_labels, probabilities)\\n            torch.save({\\n                \\"architecture\\": \\"efficientnet_b0\\", \\"classes\\": CLASSES,\\n                \\"preprocessing\\": PREPROCESSING, \\"threshold\\": threshold,\\n                \\"state_dict\\": {k: v.detach().cpu() for k, v in model.state_dict().items()},\\n                \\"epoch\\": epoch + 1, \\"seed\\": args.seed,\\n                \\"training_samples\\": len(train),\\n                \\"dataset_provenance\\": json.loads((args.data / \\"provenance.json\\").read_text())\\n                if (args.data / \\"provenance.json\\").exists() else {\\"source\\": \\"user_supplied\\"},\\n                \\"validation_metrics\\": binary_metrics(all_labels, probabilities, threshold),\\n                \\"validation_loss\\": val_loss,\\n                \\"status\\": \\"research_only_unvalidated\\",\\n            }, args.output)\\n    print(f\\"Saved {args.output}. Run evaluate.py on the held-out test split.\\")\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n"}')
for name, content in SOURCES.items():
    destination = PROJECT / name
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(content)
os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
print('Project ready:', PROJECT)

## 3. Download the Kaggle dataset
We use **EyePACS-AIROGS-light-V2, version 4**. Its publisher supplies train, validation, and test folders.
We preserve those splits and map `NRG → non_referable (0)` and `RG → referable (1)` explicitly.
This is a balanced subset, so its class mix does not represent disease prevalence in the general population.

Public downloads may work without signing in. If Kaggle requests authentication, add a Colab secret
named `KAGGLE_API_TOKEN` using the key icon in the sidebar, grant this notebook access, then rerun this cell.
You can create the token in [Kaggle settings](https://www.kaggle.com/settings/api).
Read the dataset page and its linked source terms before use or redistribution.

In [ ]:
import kagglehub
from prepare_data import DATASET
raw_path = Path(kagglehub.dataset_download(DATASET))
print("Downloaded dataset:", DATASET)
print("Dataset root:", raw_path)

## 4. Validate images and preserve the splits
This checks file contents, rejects broken or oversized images, and checks for byte-identical duplicates.
It writes a manifest with source paths and hashes. File-hash checks cannot prove that different photographs
come from different patients; the publisher's patient separation has not been independently verified.
The prepared data stays in the temporary runtime and is not uploaded to GitHub.

In [ ]:
from prepare_data import prepare
data_path = PROJECT / "data/prepared"
if data_path.exists():
    raise RuntimeError("Prepared data already exists. To reuse it, skip this cell. For a fresh run, delete the runtime first.")
provenance = prepare(raw_path, data_path)
print(json.dumps(provenance, indent=2))

## 5. Train EfficientNet-B0
We start from ImageNet weights, freeze the feature extractor, and train a two-class head for 12 epochs.
Every image is resized to **224 × 224**, converted to RGB, and normalized using ImageNet mean and standard deviation.
Frozen batch-normalization statistics stay frozen, too.
Because preprocessing is deterministic and the backbone is frozen, its features are extracted once
into memory. The classification head then trains on those features for all 12 epochs.

The best checkpoint is chosen using **validation loss**. Its routing threshold is chosen on validation
images to target at least 88% recall while maximizing specificity. Test images do not choose weights or thresholds.
Meeting the validation target does **not** mean the model will meet it on the test set or on new patients.

In [ ]:
from glaucoma.runtime import run_logged
EPOCHS = 12
BATCH_SIZE = 32
run_logged([sys.executable, "-u", "train.py", "--data", str(data_path),
                "--epochs", str(EPOCHS), "--batch-size", str(BATCH_SIZE),
                "--output", "models/best_model.pth"])

## 6. Inspect training
A falling training loss with rising validation loss can be a sign of overfitting:
the model is learning the training examples more than useful patterns.

In [ ]:
import matplotlib.pyplot as plt
history = json.loads(Path("models/best_model.history.json").read_text())
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot([r["epoch"] for r in history], [r["train_loss"] for r in history], label="Training", color="#147d70")
ax.plot([r["epoch"] for r in history], [r["validation_loss"] for r in history], label="Validation", color="#c99341")
ax.set(xlabel="Epoch", ylabel="Cross-entropy loss", title="Training and validation loss")
ax.legend()
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

## 7. Evaluate once on the held-out test set
This uses the saved threshold without tuning it on test images. The report includes sensitivity
(how many referable images it catches), specificity (how many non-referable images it leaves unflagged),
and false positives/negatives. We also time the real CPU pipeline after one warmup.
The 2-second target is measured for decoding, preprocessing, and inference; browser transfer and rendering
need a separate end-to-end check. CPU performance here describes this Colab machine, not every laptop.

In [ ]:
run_logged([sys.executable, "-u", "evaluate.py", "--data", str(data_path),
            "--model", "models/best_model.pth", "--output", "reports/evaluation.json"])
report = json.loads(Path("reports/evaluation.json").read_text())
metrics = report["metrics"]
print(f"TEST sensitivity: {metrics['sensitivity']:.1%} | specificity: {metrics['specificity']:.1%}")
print(f"CPU pipeline p95: {report['p95_pipeline_ms']:.0f} ms")
print("88% sensitivity target:", "MET on this test set" if report["sensitivity_target_met"] else "NOT MET")
print("2-second pipeline target:", "MET on this CPU" if report["latency_target_met"] else "NOT MET")
matrix = [[metrics["true_negative"], metrics["false_positive"]],
          [metrics["false_negative"], metrics["true_positive"]]]
fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(matrix, cmap="Greens", vmin=0)
for y in range(2):
    for x in range(2):
        ax.text(x, y, str(matrix[y][x]), ha="center", va="center", fontsize=20,
                color="white" if matrix[y][x] > max(map(max, matrix)) / 2 else "#193e43")
ax.set(xticks=[0, 1], yticks=[0, 1], xticklabels=["Non-referable", "Referable"],
       yticklabels=["Non-referable", "Referable"], xlabel="Predicted", ylabel="Dataset label",
       title="Held-out test confusion matrix")
plt.tight_layout()
plt.show()

## 8. Try the trained model
Upload one PNG/JPEG under 10 MB, then select **Analyze image**. This uses the trained checkpoint;
it does not generate sample scores. It runs on CPU, even if training used a GPU.
Format validation does not detect whether an image is a retina or whether it is suitable for diagnosis.

Uploads are held in this runtime's memory. Use **Clear image** when done. Clear cell outputs before
sharing the notebook, and keep **Edit → Notebook settings → Omit code cell output when saving** enabled.
Notebook outputs may contain the uploaded image. Use de-identified research images for the demo.

In [ ]:
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass
from glaucoma.notebook_ui import display_assistant
dashboard = display_assistant("models/best_model.pth")

## 9. Export the trained model and evidence
Colab runtimes are temporary. Download this archive before disconnecting.
It includes the model, its metadata, training history, evaluation report, data provenance, and exact installed package versions.
It does **not** include the dataset, your API token, or uploaded demo images.
The repository contains code and a notebook; trained artifacts are separate downloads.

In [ ]:
import zipfile
archive = PROJECT / "GlaucomAI_trained_artifacts.zip"
versions = subprocess.run([sys.executable, "-m", "pip", "freeze"], check=True, capture_output=True, text=True).stdout
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as bundle:
    for name in ["models/best_model.pth", "models/best_model.history.json", "reports/evaluation.json",
                 "data/prepared/provenance.json"]:
        bundle.write(name, arcname=name)
    bundle.writestr("environment.txt", versions)
print("Artifacts ready:", archive.name)
from IPython.display import FileLink, display
display(FileLink(archive.name))

### Download to your computer (run this cell when ready)
Colab's download helper opens a browser download. Outside Colab, use the archive link above.

In [ ]:
# Uncomment these two lines in Colab when you want to download the archive:
# from google.colab import files
# files.download(str(archive))

## Reopen a trained model without retraining
In a fresh runtime, run steps 1–2. Upload your trusted `best_model.pth` with the Colab Files sidebar
into `/content/GlaucomAI/models/` (create the `models` folder if needed), then run step 8.
Only load checkpoints you trust. Do not rerun steps 3–7 just to show the demo.

## What this prototype proves—and what it does not
This is a binary image classifier. It does **not** segment the optic cup/disc, measure cup-to-disc ratio,
establish the cause of a prediction, or diagnose glaucoma. Softmax scores are not calibrated clinical risks.
Publisher splits and exact-duplicate checks are useful, but patient separation and cross-dataset performance
still need independent verification. The 88% recall and 2-second latency values are targets until measured.

**Sources:** [dataset and citations](https://www.kaggle.com/datasets/deathtrooper/glaucoma-dataset-eyepacs-airogs-light-v2/versions/4),
[KaggleHub authentication](https://github.com/Kaggle/kagglehub#authenticate),
[EfficientNet-B0](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.efficientnet_b0.html),
[Colab limits and sharing](https://research.google.com/colaboratory/faq.html).